# METRIC-RES-001 — checkpoint-only identity-resolution audit (Kaggle T4 x2)

**What this decides.** S5's 24-arm mechanism campaign returned `INCONCLUSIVE_AT_ZERO_BASELINE` (identity 0.0–0.008 in every arm). The returned-bundle audit prescribed one next action: re-audit the **preserved checkpoints** at token resolution to learn whether identity is *forming-but-unmeasured* or *genuinely absent*.

**This trains nothing.** No optimizer, no backward, no parameter mutation. It loads S5 weights read-only and re-scores them at token resolution.

**Why Kaggle.** The audit is 16 model evaluations. On Colab it is 20–40 min of your daily T4 allowance. On Kaggle T4 x2 it is ~10–20 min of the **30 GPU-h/week** pool and shards cleanly across both GPUs, so the Colab budget stays free for the baseline gate and the tie-role pilot.

**Before you run**
1. Settings → Accelerator → **GPU T4 x2**, Internet **ON** (the notebook clones the repo).
2. Attach the S5 Output as Kaggle Input so `/kaggle/input/.../FORMATION_MUX_001/CS-MECH-002/<ARM>/S<1-4>/resume.pt` and `PUBLIC_SURFACE_MANIFEST.json` are present. The S5 result ZIP does **not** contain the checkpoints; they are still in the original S5 run's Output tree.
3. Expected wall time: **10–20 min**. Well inside the 12h session cap.

**Quota architecture:** Cell 2 (GPU) does inference only. Cell 3 (CPU, Accelerator=None) applies the decision tree and packages — zero GPU-h.

In [ ]:
# CELL 1 — fetch pinned code, verify identity, no torch in this kernel
import os
# Architecture: allocator is set before any GPU process starts, and this
# kernel never imports torch, so it holds no CUDA context on either T4.
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ['PYTHONUNBUFFERED'] = '1'
import json, pathlib, shutil, subprocess, sys

REPO = pathlib.Path('/kaggle/working/An-Ra-the-new-AGI')
REMOTE = 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
BRANCH = 'cymek-next-core-architecture'
EXECUTION_COMMIT = 'REPLACE_AFTER_COMMIT'
OPERATOR_BLOB = 'REPLACE_AFTER_COMMIT'
PREREG_BLOB = 'REPLACE_AFTER_COMMIT'
OP_COPY = pathlib.Path('/kaggle/working/metric_resolution_audit_v1.py')

if REPO.exists() and not (REPO/'.git').exists():
    shutil.rmtree(REPO)
if not REPO.exists():
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REMOTE,str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','-q',EXECUTION_COMMIT],check=True)
head = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
assert head==EXECUTION_COMMIT,(head,EXECUTION_COMMIT)
def blob(rel):
    return subprocess.check_output(['git','-C',str(REPO),'hash-object',rel],text=True).strip()
assert blob('tools/metric_resolution_audit_v1.py')==OPERATOR_BLOB
assert blob('docs/cymek/experiments/METRIC-RES-001/PREREGISTRATION.json')==PREREG_BLOB
shutil.copy2(REPO/'tools/metric_resolution_audit_v1.py',OP_COPY)
try:
    __import__('tokenizers')
except ImportError:
    subprocess.run([sys.executable,'-m','pip','install','-q','tokenizers'],check=True)
# Hardware gate via nvidia-smi only: no torch import, no CUDA context.
smi = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader,nounits'],capture_output=True,text=True)
gpus = [l.strip() for l in smi.stdout.strip().splitlines() if l.strip()] if smi.returncode==0 else []
print('GPUs visible:', gpus)
if len(gpus)!=2 or not all('T4' in g for g in gpus):
    raise RuntimeError('AUDIT BLOCKED: Settings -> Accelerator -> GPU T4 x2 (observed: '+str(gpus)+')')
# Locate the S5 checkpoint tree across attached inputs.
candidates = sorted(pathlib.Path('/kaggle/input').rglob('FORMATION_MUX_001/PUBLIC_SURFACE_MANIFEST.json'))
if not candidates:
    raise RuntimeError('Attach the S5 run Output as Kaggle Input (it holds CS-MECH-002/*/S*/resume.pt + PUBLIC_SURFACE_MANIFEST.json)')
CKPT_ROOT = candidates[0].parent
n_ck = len(list(CKPT_ROOT.rglob('CS-MECH-002/*/S*/resume.pt')))
print('checkpoint root:', CKPT_ROOT)
print('resume.pt found:', n_ck, '(expect 16 for 4 arms x 4 seeds)')
if n_ck < 16:
    raise RuntimeError('Incomplete checkpoint set: '+str(n_ck)+'/16. Attach the full S5 Output.')
OUT = pathlib.Path('/kaggle/working/METRIC_RES_001')
print('READY')

In [ ]:
# CELL 2 — GPU stage: 2-way shard, one process per T4 (no DDP)
import pathlib, subprocess, sys, time
OUT = pathlib.Path('/kaggle/working/METRIC_RES_001')
OUT.mkdir(parents=True, exist_ok=True)
base = [sys.executable,'-u',str(OP_COPY),'--repo',str(REPO),'--out',str(OUT),
        '--checkpoints',str(CKPT_ROOT),'--device','cuda']
cmds = [base+['--shard-index',str(i),'--num-shards','2'] for i in (0,1)]
envs = [dict(os.environ, CUDA_VISIBLE_DEVICES=str(i), OMP_NUM_THREADS='1', MKL_NUM_THREADS='1') for i in (0,1)]
logdir = OUT/'_shard_logs'; logdir.mkdir(parents=True, exist_ok=True)
print('Starting 2 GPU shards. Expect 10-20 min.', flush=True)
t0 = time.monotonic()
procs = []
for i, cmd in enumerate(cmds):
    handle = (logdir/f'shard{i}.log').open('w', encoding='utf-8', buffering=1)
    print(f'GPU{i} <- shard {i}', flush=True)
    procs.append((subprocess.Popen(cmd, cwd=REPO, env=envs[i], stdout=handle, stderr=subprocess.STDOUT), handle))
codes = []
try:
    for p, _ in procs:
        codes.append(p.wait())
finally:
    for _, h in procs:
        try: h.close()
        except Exception: pass
print('SHARD RETURN CODES:', codes, '| elapsed min:', round((time.monotonic()-t0)/60,1), flush=True)
if codes != [0,0]:
    for i in (0,1):
        lg = logdir/f'shard{i}.log'
        if lg.exists(): print(f'--- shard{i} tail ---\n'+lg.read_text()[-2000:])
    raise SystemExit('Shard failed closed. Partial receipts are preserved in /kaggle/working/METRIC_RES_001.')
print('arm receipts:', len(list(OUT.glob('ARM_*.json'))), '(expect 16)')

In [ ]:
# CELL 3 — CPU stage (FREE): switch Accelerator to None, then run.
# Applies the frozen decision tree, re-adjudicates the S5 contrasts in
# token space, packages, and syncs. Imports no torch.
import json, pathlib, subprocess, sys
OUT = pathlib.Path('/kaggle/working/METRIC_RES_001')
cmd = [sys.executable,'-u',str(OP_COPY),'--repo',str(REPO),'--out',str(OUT),
       '--checkpoints',str(CKPT_ROOT),'--aggregate-only']
if subprocess.run(cmd, cwd=REPO).returncode != 0:
    raise SystemExit('Aggregate failed closed. Do not interpret partial results.')
d = json.loads((OUT/'DECISION.json').read_text())
print()
print('DECISION:', d['decision']['decision'])
print('MEASURED:', json.dumps(d['decision']['measured'], indent=2))
for row in d['contrast_readjudication']['contrasts']:
    print('CONTRAST', row['contrast'], '->', row['verdict'], '| mean_delta', row['mean_delta'], '| 3of4', row['sign_consistent_3of4'])
print('CLAIM CEILING:', d['claim_ceiling'])
b = pathlib.Path('/kaggle/working/METRIC_RES_001_RESULTS.zip')
print('RESULT ZIP:', b, '| exists', b.exists())